In [ ]:
!pip install transformers torch onnx onnxruntime optimum onnxscript

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
import json

In [ ]:
# Step 1: Define the model architecture
class TinyBERTDualClassifier(nn.Module):
    def __init__(self, num_module_labels, num_date_labels, dropout_rate=0.1):
        super(TinyBERTDualClassifier, self).__init__()
        self.encoder = AutoModel.from_pretrained("JayShah07/tinybert-dual-classifier")
        self.hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(p=dropout_rate)
        self.module_classifier = nn.Linear(self.hidden_size, num_module_labels)
        self.date_classifier = nn.Linear(self.hidden_size, num_date_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        module_logits = self.module_classifier(cls_output)
        date_logits = self.date_classifier(cls_output)
        return module_logits, date_logits

# Step 2: Load the model weights
print("Loading model...")
classifier_config = torch.hub.load_state_dict_from_url(
    "https://huggingface.co/JayShah07/tinybert-dual-classifier/resolve/main/classifier_heads.pt",
    map_location=torch.device('cpu')
)

Loading model...
Downloading: "https://huggingface.co/JayShah07/tinybert-dual-classifier/resolve/main/classifier_heads.pt" to /root/.cache/torch/hub/checkpoints/classifier_heads.pt


100%|██████████| 18.8k/18.8k [00:00<00:00, 25.8MB/s]


In [ ]:
model = TinyBERTDualClassifier(
    num_module_labels=6,
    num_date_labels=7,
    dropout_rate=0.0  # Set to 0 for inference
)

model.module_classifier.load_state_dict(classifier_config['module_classifier'])
model.date_classifier.load_state_dict(classifier_config['date_classifier'])
model.eval()

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("JayShah07/tinybert-dual-classifier")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/57.4M [00:00<?, ?B/s]

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
print("Exporting to ONNX...")

# Create dummy inputs
dummy_text = "Show my holdings for this month"
dummy_inputs = tokenizer(
    dummy_text,
    return_tensors='pt',
    padding='max_length',
    truncation=True,
    max_length=128
)

input_ids = dummy_inputs['input_ids']
attention_mask = dummy_inputs['attention_mask']

# Export to ONNX
torch.onnx.export(
    model,
    (input_ids, attention_mask),
    "tinybert_dual_classifier.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input_ids', 'attention_mask'],
    output_names=['module_logits', 'date_logits'],
    dynamic_axes={
        'input_ids': {0: 'batch_size'},
        'attention_mask': {0: 'batch_size'},
        'module_logits': {0: 'batch_size'},
        'date_logits': {0: 'batch_size'}
    },
    keep_initializers_as_inputs=True  # ❗ sometimes fixes shape/value_info mismatches
)

print("ONNX model exported successfully!")


/tmp/ipython-input-3815283221.py:17: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1219 12:26:11.996000 217 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Exporting to ONNX...
[torch.onnx] Obtain model graph for `TinyBERTDualClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TinyBERTDualClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 21 of general pattern rewrite rules.
ONNX model exported successfully!


In [ ]:
print("Quantizing model for faster inference...")
quantize_dynamic(
    "tinybert_dual_classifier.onnx",
    "tinybert_dual_classifier_quantized.onnx",
    weight_type=QuantType.QUInt8
)
print("Quantized model created!")

# ==================== SAVE TOKENIZER VOCABULARY ====================
print("Saving tokenizer data...")

# Save vocabulary
vocab = tokenizer.get_vocab()
with open('vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

# Save tokenizer config
tokenizer_config = {
    'max_length': 128,
    'padding': 'max_length',
    'truncation': True,
    'vocab_size': len(vocab),
    'cls_token': tokenizer.cls_token,
    'sep_token': tokenizer.sep_token,
    'pad_token': tokenizer.pad_token,
    'unk_token': tokenizer.unk_token,
    'cls_token_id': tokenizer.cls_token_id,
    'sep_token_id': tokenizer.sep_token_id,
    'pad_token_id': tokenizer.pad_token_id,
    'unk_token_id': tokenizer.unk_token_id,
}

with open('tokenizer_config.json', 'w') as f:
    json.dump(tokenizer_config, f, indent=2)

# Save label mappings
labels_config = {
    'module_labels': ['holdings', 'capital_gains', 'scheme_wise_returns',
                      'investment_account_wise_returns', 'portfolio_update', 'None_module'],
    'date_labels': ['Current Year', 'Previous Year', 'Daily', 'Monthly',
                    'Weekly', 'Yearly', 'None_date']
}

with open('labels.json', 'w') as f:
    json.dump(labels_config, f, indent=2)


Quantizing model for faster inference...
Quantized model created!
Saving tokenizer data...


In [ ]:
print("\nVerifying ONNX model...")
import onnxruntime as ort

# Test original model
ort_session = ort.InferenceSession("tinybert_dual_classifier.onnx")
ort_inputs = {
    'input_ids': input_ids.numpy(),
    'attention_mask': attention_mask.numpy()
}
ort_outputs = ort_session.run(None, ort_inputs)

print(f"Module prediction: {ort_outputs[0].argmax()}")
print(f"Date prediction: {ort_outputs[1].argmax()}")

# Test quantized model
ort_session_q = ort.InferenceSession("tinybert_dual_classifier_quantized.onnx")
ort_outputs_q = ort_session_q.run(None, ort_inputs)

print(f"\nQuantized - Module prediction: {ort_outputs_q[0].argmax()}")
print(f"Quantized - Date prediction: {ort_outputs_q[1].argmax()}")



Verifying ONNX model...
Module prediction: 0
Date prediction: 3

Quantized - Module prediction: 0
Quantized - Date prediction: 3


In [ ]:
import os
print("\n" + "="*50)
print("FILE SIZES:")
print("="*50)
original_size = os.path.getsize("tinybert_dual_classifier.onnx") / (1024*1024)
quantized_size = os.path.getsize("tinybert_dual_classifier_quantized.onnx") / (1024*1024)
print(f"Original ONNX model: {original_size:.2f} MB")
print(f"Quantized ONNX model: {quantized_size:.2f} MB")
print(f"Size reduction: {((original_size - quantized_size) / original_size * 100):.1f}%")

print("\n" + "="*50)
print("DOWNLOAD THESE FILES:")
print("="*50)
print("1. tinybert_dual_classifier_quantized.onnx (recommended)")
print("2. vocab.json")
print("3. tokenizer_config.json")
print("4. labels.json")
print("\nOptional:")
print("5. tinybert_dual_classifier.onnx (if you want non-quantized version)")
print("="*50)


FILE SIZES:
Original ONNX model: 0.04 MB
Quantized ONNX model: 13.69 MB
Size reduction: -38867.4%

DOWNLOAD THESE FILES:
1. tinybert_dual_classifier_quantized.onnx (recommended)
2. vocab.json
3. tokenizer_config.json
4. labels.json

Optional:
5. tinybert_dual_classifier.onnx (if you want non-quantized version)
